# Investigación auxiliar: GUIDs con múltiples PIDs

Objetivo: identificar los 13 GUIDs de `ProcessGuid` que aparecen asociados a más de un `ProcessId`,
entender qué PIDs tiene cada GUID, y determinar qué EventIDs son los que producen la discrepancia.

Este notebook es exploratorio — el código final se integrará en `8_sysmon_csv_exploratory_analysis.ipynb`.

In [ ]:
import pandas as pd

TARGET_PATH = "../dataset/run-01-apt-1/02_sysmon-run-01.csv"

df = pd.read_csv(
    TARGET_PATH,
    low_memory=False,
    dtype={
        'ProcessId': 'Int64', 'SourcePort': 'Int64', 'DestinationPort': 'Int64',
        'SourceProcessId': 'Int64', 'ParentProcessId': 'Int64',
        'SourceThreadId': 'Int64', 'TargetProcessId': 'Int64',
        'ProcessGuid': 'string', 'SourceProcessGUID': 'string',
        'TargetProcessGUID': 'string', 'ParentProcessGuid': 'string',
        'Computer': 'category', 'Protocol': 'category', 'EventType': 'category'
    }
)

print(f"Dataset cargado: {len(df):,} filas x {len(df.columns)} columnas")

In [ ]:
# Paso 1: identificar los GUIDs con más de un PID

valid = df[df['ProcessGuid'].notna() & df['ProcessId'].notna()]

# Por cada GUID, contar cuántos PIDs distintos tiene
pids_per_guid = valid.groupby('ProcessGuid')['ProcessId'].nunique()
multi_pid_guids = pids_per_guid[pids_per_guid > 1]

print(f"GUIDs con más de un PID: {len(multi_pid_guids)}")
print()
print(multi_pid_guids.sort_values(ascending=False))

In [ ]:
# Paso 2: para cada GUID conflictivo, mostrar qué PIDs tiene y en qué EventIDs aparece

for guid in multi_pid_guids.index:
    subset = valid[valid['ProcessGuid'] == guid][['EventID', 'ProcessId', 'Computer', 'Image']]
    pids = subset['ProcessId'].unique()
    eids = subset['EventID'].unique()
    print(f"GUID: {guid}")
    print(f"  PIDs: {sorted(pids)}")
    print(f"  EventIDs: {sorted(eids)}")
    print(subset.drop_duplicates(subset=['EventID', 'ProcessId']).to_string(index=False))
    print()

In [ ]:
# Paso 3: distribución de EventIDs en los registros conflictivos

conflicting_rows = valid[valid['ProcessGuid'].isin(multi_pid_guids.index)]
print(f"Total filas involucradas: {len(conflicting_rows)}")
print()
print("Distribución por EventID:")
print(conflicting_rows['EventID'].value_counts().sort_index())

---
## Análisis sobre todos los APT runs

Verifica si el GUID nulo (`00000000-0000-0000-0000-000000000000`) es el único conflictivo
en todos los runs, y qué EventIDs lo reciben en cada uno.

Solo se procesan los CSVs con prefijo `02_` (pipeline actualizado). Los runs sin ese archivo se saltan.

In [5]:
import os
import glob
import pandas as pd

DATASET_ROOT = "/home/researcher/Research/phd-thesis/fullapt2025/dataset"
NULL_GUID    = "00000000-0000-0000-0000-000000000000"

# Solo columnas necesarias para ahorrar memoria
COLS   = ['EventID', 'ProcessGuid', 'ProcessId']
DTYPES = {'ProcessId': 'Int64', 'ProcessGuid': 'string'}

summary_rows = []   # una fila por run
null_guid_eids = {} # run -> set de EventIDs con null GUID
other_guids    = {} # run -> {guid: n_pids} para GUIDs no-nulos con >1 PID

run_dirs = sorted(glob.glob(os.path.join(DATASET_ROOT, "run-*")))

for run_dir in run_dirs:
    run_name = os.path.basename(run_dir)
    csv_files = glob.glob(os.path.join(run_dir, "02_sysmon-run-*.csv"))

    if not csv_files:
        print(f"[SKIP] {run_name}: no 02_sysmon CSV found")
        continue

    csv_path = csv_files[0]

    try:
        df = pd.read_csv(csv_path, low_memory=False,
                         usecols=COLS, dtype=DTYPES)

        valid = df[df['ProcessGuid'].notna() & df['ProcessId'].notna()]
        pids_per_guid = valid.groupby('ProcessGuid')['ProcessId'].nunique()
        multi = pids_per_guid[pids_per_guid > 1]

        # null GUID
        has_null = NULL_GUID in multi.index
        null_pids = int(multi.get(NULL_GUID, 0))
        null_rows = int((valid['ProcessGuid'] == NULL_GUID).sum()) if has_null else 0
        eids_with_null = sorted(
            valid[valid['ProcessGuid'] == NULL_GUID]['EventID'].unique().tolist()
        ) if has_null else []

        # GUIDs no-nulos con >1 PID
        other = multi[multi.index != NULL_GUID]

        summary_rows.append({
            'run':            run_name,
            'total_rows':     len(df),
            'multi_pid_guids': len(multi),
            'has_null_guid':  has_null,
            'null_guid_pids': null_pids,
            'null_guid_rows': null_rows,
            'null_guid_eids': str(eids_with_null),
            'other_guids_count': len(other),
        })

        if len(other) > 0:
            other_guids[run_name] = other.to_dict()

        print(f"[OK] {run_name}: {len(multi)} GUID(s) con >1 PID | "
              f"null_guid={'YES' if has_null else 'no'} ({null_rows} filas) | "
              f"other={len(other)} | EIDs null: {eids_with_null}")

    except Exception as e:
        print(f"[ERROR] {run_name}: {e}")
        summary_rows.append({'run': run_name, 'error': str(e)})

print("\nFINISHED")

[OK] run-01-apt-1: 1 GUID(s) con >1 PID | null_guid=YES (36 filas) | other=0 | EIDs null: [3, 7, 13]
[OK] run-02-apt-1: 1 GUID(s) con >1 PID | null_guid=YES (208 filas) | other=0 | EIDs null: [3, 7, 9, 12, 13]
[OK] run-03-apt-1: 1 GUID(s) con >1 PID | null_guid=YES (65 filas) | other=0 | EIDs null: [3, 7]
[OK] run-04-apt-1: 1 GUID(s) con >1 PID | null_guid=YES (23 filas) | other=0 | EIDs null: [3]
[OK] run-05-apt-1: 1 GUID(s) con >1 PID | null_guid=YES (284 filas) | other=0 | EIDs null: [3, 7, 12, 13]
[OK] run-06-apt-1: 1 GUID(s) con >1 PID | null_guid=YES (244 filas) | other=0 | EIDs null: [3, 7, 9, 12, 13]
[OK] run-07-apt-1: 1 GUID(s) con >1 PID | null_guid=YES (141 filas) | other=0 | EIDs null: [3, 7, 9, 12, 13]
[OK] run-08-apt-1: 1 GUID(s) con >1 PID | null_guid=YES (113 filas) | other=0 | EIDs null: [3, 7, 12, 13]
[OK] run-09-apt-1: 1 GUID(s) con >1 PID | null_guid=YES (120 filas) | other=0 | EIDs null: [3, 5, 7, 9, 12, 13]
[OK] run-10-apt-1: 1 GUID(s) con >1 PID | null_guid=YES (

In [6]:
# Resumen tabular
summary_df = pd.DataFrame(summary_rows)
print(summary_df[['run','total_rows','multi_pid_guids','has_null_guid',
                   'null_guid_pids','null_guid_rows','null_guid_eids',
                   'other_guids_count']].to_string(index=False))

print("\n--- GUIDs no-nulos con >1 PID (si los hay) ---")
if other_guids:
    for run, guids in other_guids.items():
        print(f"  {run}: {guids}")
else:
    print("  Ninguno — el null GUID es el único caso en todos los runs procesados.")

         run  total_rows  multi_pid_guids  has_null_guid  null_guid_pids  null_guid_rows       null_guid_eids  other_guids_count
run-01-apt-1      363657                1           True              14              36           [3, 7, 13]                  0
run-02-apt-1      570078                1           True              51             208    [3, 7, 9, 12, 13]                  0
run-03-apt-1      196409                1           True              22              65               [3, 7]                  0
run-04-apt-1       65265                1           True               9              23                  [3]                  0
run-05-apt-1     1266782                1           True              63             284       [3, 7, 12, 13]                  0
run-06-apt-1     1574322                1           True              61             244    [3, 7, 9, 12, 13]                  0
run-07-apt-1      656322                1           True              47             141    [3, 7